# USANDO UNA RED NEURONAL RECURRENTE

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Parámetros

In [2]:
input_size = 1     # Cada número es un escalar
hidden_size = 16   # Tamaño del estado oculto
output_size = 1    # Queremos predecir un solo número
num_layers = 1

# Modelo RNN personalizado

In [3]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(num_layers, x.size(0), hidden_size)  # estado inicial oculto
        out, _ = self.rnn(x, h0)  # salida completa de la RNN
        out = self.fc(out[:, -1, :])  # tomamos la salida del último paso
        return out

In [4]:
# Instanciar el modelo
model = SimpleRNN(input_size, hidden_size, output_size, num_layers)

In [5]:
# Pérdida y optimizador
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Datos de entrenamiento: secuencia de entrada y el siguiente número

In [6]:
X = torch.tensor([[[1.0], [2.0], [3.0]]])  # shape: (batch=1, seq_len=3, input_size=1)
y = torch.tensor([[4.0]])  # shape: (batch=1, output_size=1)

# Entrenamiento

In [7]:
for epoch in range(300):
    output = model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}, Predicción: {output.item():.4f}')

Epoch 0, Loss: 15.6477, Predicción: 0.0443
Epoch 50, Loss: 0.0027, Predicción: 3.9484
Epoch 100, Loss: 0.0003, Predicción: 3.9826
Epoch 150, Loss: 0.0000, Predicción: 4.0010
Epoch 200, Loss: 0.0000, Predicción: 3.9999
Epoch 250, Loss: 0.0000, Predicción: 4.0000


# RED NEURONAL USANDO MECANISMOS DE ATENCIÓN

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Función de atención escalar (dot-product)

In [9]:
def attention(query, key, value):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, value)
    return output, attn_weights

# CREAMOS EL MODELO DE RED NEURONAL USANDO LA FUNCIÓN DE ATENCIÓN

In [10]:
class AttentionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(AttentionModel, self).__init__()
        self.query_layer = nn.Linear(input_dim, hidden_dim)
        self.key_layer = nn.Linear(input_dim, hidden_dim)
        self.value_layer = nn.Linear(input_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, 1)  # Para predecir un escalar

    def forward(self, x):
        Q = self.query_layer(x)  # (batch, seq_len, hidden)
        K = self.key_layer(x)
        V = self.value_layer(x)

        attended, attn_weights = attention(Q, K, V)  # (batch, seq_len, hidden)
        out = self.fc_out(attended[:, -1, :])  # Tomamos solo la última posición
        return out

In [11]:
# Datos de entrada
X = torch.tensor([[[1.0], [2.0], [3.0]]])  # shape: (1, 3, 1)
y = torch.tensor([[4.0]])

In [12]:
# Parámetros
input_dim = 1
hidden_dim = 16
model = AttentionModel(input_dim, hidden_dim)

In [13]:
# Entrenamiento
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [14]:
for epoch in range(300):
    output = model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Predicción: {output.item():.4f}")

Epoch 0 | Loss: 29.5261 | Predicción: -1.4338
Epoch 50 | Loss: 0.0515 | Predicción: 3.7731
Epoch 100 | Loss: 0.0000 | Predicción: 3.9968
Epoch 150 | Loss: 0.0000 | Predicción: 3.9988
Epoch 200 | Loss: 0.0000 | Predicción: 4.0001
Epoch 250 | Loss: 0.0000 | Predicción: 4.0000
